# 02 Cleaning

This notebook performs the Person A workflow for the Olist project: cleaning the raw tables, validating joins, engineering order-level features, and exporting processed datasets for analysis and Tableau.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.etl_pipeline import build_clean_tables, write_outputs, build_summary


In [2]:
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

raw_tables, clean_tables = build_clean_tables(RAW_DIR)
print('Loaded raw tables:', ', '.join(raw_tables.keys()))
print('Prepared clean tables:', ', '.join(clean_tables.keys()))


Loaded raw tables: orders, customers, geolocation, order_items, payments, reviews, products, sellers, translation
Prepared clean tables: orders, customers, geo_lookup, order_items_fact, order_items, payments, reviews, products, sellers, translation


In [3]:
# Validate the key cleaning pressure points in the raw data.
quality_snapshot = {
    'orders_missing_delivery_date': int(raw_tables['orders']['order_delivered_customer_date'].isna().sum()),
    'orders_missing_carrier_date': int(raw_tables['orders']['order_delivered_carrier_date'].isna().sum()),
    'orders_missing_approval_date': int(raw_tables['orders']['order_approved_at'].isna().sum()),
    'reviews_missing_comment_message': int(raw_tables['reviews']['review_comment_message'].isna().sum()),
    'products_missing_category_name': int(raw_tables['products']['product_category_name'].isna().sum()),
}
quality_snapshot


{'orders_missing_delivery_date': 2965,
 'orders_missing_carrier_date': 1783,
 'orders_missing_approval_date': 160,
 'reviews_missing_comment_message': 58247,
 'products_missing_category_name': 610}

In [5]:
# Inspect the cleaned/aggregated intermediate tables that feed the final joins.
for name in ['customers', 'sellers', 'products', 'order_items', 'payments', 'reviews']:
    df = clean_tables[name]
    print(f'=== {name} === shape={df.shape}')
    display(df.head(3))


=== customers === shape=(99441, 9)


,customer_id,customer_unique_id,zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,-20.499273,-47.396658,franca,SP,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,-23.728396,-46.542250,sao bernardo do campo,SP,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,-23.531309,-46.656690,sao paulo,SP,sao paulo,SP


=== sellers === shape=(3095, 8)


,seller_id,zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,-22.893317,-47.060596,campinas,SP,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,-22.383375,-46.948142,mogi-guacu,SP,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,-22.909446,-43.180240,rio de janeiro,RJ,rio de janeiro,RJ


=== products === shape=(32951, 13)


...

,order_id,item_count,unique_products,unique_sellers,total_price,total_freight,avg_item_price,max_item_price,primary_product_category,primary_seller_state,contains_missing_product_metadata
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.9,13.29,58.9,58.9,cool_stuff,SP,0
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.9,19.93,239.9,239.9,pet_shop,SP,0
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.0,17.87,199.0,199.0,furniture_decor,MG,0


=== payments === shape=(99440, 6)


,order_id,payment_records,payment_value_total,payment_installments_max,payment_type_primary,payment_types_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2,credit_card,1
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3,credit_card,1
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5,credit_card,1


=== reviews === shape=(98673, 5)


,order_id,review_score,review_has_comment,review_created_at,review_answered_at
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,4.0,0,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,5.0,1,2018-01-23,2018-01-23 16:06:31


In [6]:
# Export all processed outputs.
exports = write_outputs(PROCESSED_DIR, clean_tables)
summary = build_summary(raw_tables, clean_tables, exports)
(PROCESSED_DIR / 'etl_summary.json').write_text(json.dumps(summary, indent=2, default=str))

pd.DataFrame([
    {'file_name': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in exports.items()
]).sort_values('file_name')


,file_name,rows,columns
3,olist_customers_clean.csv,99441,9
1,olist_delivered_orders_master.csv,96478,50
2,olist_order_items_enriched.csv,112650,26
0,olist_orders_master.csv,99441,50
5,olist_products_clean.csv,32951,13
4,olist_sellers_clean.csv,3095,8
6,olist_zip_geolocation_lookup.csv,19015,5


In [7]:
# Load the two most important handoff files for validation.
orders_master = pd.read_csv(PROCESSED_DIR / 'olist_orders_master.csv')
delivered_master = pd.read_csv(PROCESSED_DIR / 'olist_delivered_orders_master.csv')

print('orders_master shape:', orders_master.shape)
print('delivered_master shape:', delivered_master.shape)
print('delivered share:', round(delivered_master.shape[0] / orders_master.shape[0], 4))


orders_master shape: (99441, 50)
delivered_master shape: (96478, 50)
delivered share: 0.9702


In [8]:
# Confirm the most important fields that Person B and Tableau will rely on.
orders_master[[
    'order_id', 'order_status', 'customer_state', 'primary_product_category',
    'order_value', 'delivery_days', 'is_late_delivery', 'review_score',
    'payment_type_primary', 'order_purchase_month'
]].head(10)


,order_id,order_status,customer_state,primary_product_category,order_value,delivery_days,is_late_delivery,review_score,payment_type_primary,order_purchase_month
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,housewares,38.71,8.436574,0,4.0,voucher,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,perfumery,141.46,13.782037,0,4.0,boleto,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,auto,179.12,9.394213,0,5.0,credit_card,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,pet_shop,72.20,13.208750,0,5.0,credit_card,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,stationery,28.62,2.873877,0,5.0,credit_card,2018-02
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,PR,auto,175.26,16.542245,0,4.0,credit_card,2017-07
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,RS,unknown,65.95,NaN,0,2.0,credit_card,2017-04
7,6514b8ad8028c9f2cc2374ded245783f,delivered,RJ,auto,75.16,9.989826,0,5.0,credit_card,2017-05
8,76c6e866289321a7c93b82b54852dc33,delivered,RS,furniture_decor,35.95,9.818762,0,1.0,boleto,2017-01
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,SP,office_furniture,169.76,18.221852,0,5.0,credit_card,2017-07


In [9]:
# Quick diagnostic metrics for downstream teammates.
metrics = {
    'order_status_distribution': orders_master['order_status'].value_counts().to_dict(),
    'late_delivery_rate_delivered': round(delivered_master['is_late_delivery'].mean(), 4),
    'avg_review_score_delivered': round(delivered_master['review_score'].mean(), 4),
    'orders_with_review_comments': int(orders_master['review_has_comment'].fillna(0).sum()),
    'orders_with_missing_product_metadata': int(orders_master['contains_missing_product_metadata'].fillna(0).sum()),
}
metrics


{'order_status_distribution': {'delivered': 96478,
  'shipped': 1107,
  'canceled': 625,
  'unavailable': 609,
  'invoiced': 314,
  'processing': 301,
  'created': 5,
  'approved': 2},
 'late_delivery_rate_delivered': np.float64(0.0811),
 'avg_review_score_delivered': np.float64(4.1284),
 'orders_with_review_comments': 40836,
 'orders_with_missing_product_metadata': 1452}

## Cleaning Decisions Applied

- Standardized all column names to `snake_case`.
- Parsed datetime fields before creating delivery and timing features.
- Aggregated the geolocation table to a zip-prefix lookup to avoid many-to-many joins.
- Filled missing review text with placeholders and created a `review_has_comment` flag.
- Filled missing product metadata numerics with median values and created a `contains_missing_product_metadata` flag.
- Aggregated order items, payments, and reviews to one row per order before building Tableau-ready outputs.
- Exported both an all-orders master file and a delivered-orders-only file for different downstream use cases.


## Handoff Guidance

- Person B should use `data/processed/olist_delivered_orders_master.csv` for KPI analysis, EDA, and hypothesis testing.
- Tableau teammates should start from `data/processed/olist_orders_master.csv` to keep one row per order and avoid join duplication.
- The documentation teammate can use `data/processed/etl_summary.json` and `docs/person_a_handoff.md` for row counts, cleaning notes, and data-quality discussion.
